
# Phase 2 — Time Series EDA

Exploratory analysis of **Phase 1** output (`usa_fortnight_features.csv`).

**Input:** fortnight field features for Texas, Louisiana, Florida  
**Focus:** time-series vegetation indices, distributions, correlations, and data quality — adapted from the Sugarcane monitoring EDA patterns, but for **yield regression** data.

Run in Google Colab after Phase 1, or locally with the CSV in the same folder.


In [ ]:

# --- CONFIG ---
INPUT_CSV = 'usa_fortnight_features.csv'

# Field to highlight in single-field time-series plots (change after loading)
SAMPLE_FIELD_ID = 1

# Main vegetation indices for plots (matches Phase 1 output)
VEG_FEATURES = ['NDVI', 'EVI', 'GNDVI', 'SAVI']
EXTRA_FEATURES = ['MOISTURE', 'LAI']

# Correlation heatmap uses a focused feature set (keeps plot readable)
CORR_FEATURES = [
    'NDVI', 'EVI', 'GNDVI', 'SAVI', 'MSAVI', 'VCI',
    'MOISTURE', 'LAI', 'NDMI_8', 'NDWI_8', 'Elevation'
]


In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='husl')
%matplotlib inline

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not os.path.exists(INPUT_CSV):
  if IN_COLAB:
    print(f'{INPUT_CSV} not found — upload Phase 1 output:')
    uploaded = files.upload()
    INPUT_CSV = next(iter(uploaded))
  else:
    raise FileNotFoundError(f'Place {INPUT_CSV} next to this notebook or set INPUT_CSV')

df = pd.read_csv(INPUT_CSV)
print('Shape:', df.shape)
df.head()


In [ ]:

META_COLS = ['field_id', 'state', 'year', 'fortnight_start', 'fortnight_end', 'state_annual_yield', 'Area', 's2_scene_count']

# Parse fortnight dates and sort for time-series plots
df['fortnight_start'] = pd.to_datetime(df['fortnight_start'])
df['fortnight_end'] = pd.to_datetime(df['fortnight_end'])
df = df.sort_values(['field_id', 'fortnight_start']).reset_index(drop=True)

print('Date range:', df['fortnight_start'].min().date(), '→', df['fortnight_start'].max().date())
print('States:', sorted(df['state'].dropna().unique()))
print('Years:', sorted(df['year'].dropna().unique()))
print('Unique fields:', df['field_id'].nunique())
print('Rows per field (median):', df.groupby('field_id').size().median())



## 1. Dataset overview & missing data


In [ ]:

overview = pd.DataFrame({
    'rows': [len(df)],
    'fields': [df['field_id'].nunique()],
    'states': [df['state'].nunique()],
    'years': [df['year'].nunique()],
    'fortnights': [df['fortnight_start'].nunique()],
    'duplicate_field_date': [df.duplicated(['field_id', 'fortnight_start']).sum()],
})
overview.T


In [ ]:

# Missing values on key columns
check_cols = META_COLS + VEG_FEATURES + EXTRA_FEATURES
missing = df[check_cols].isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct}).query('missing > 0')


In [ ]:

# Missing-data heatmap: % missing per feature × year (focused feature set)
heatmap_features = [c for c in (VEG_FEATURES + EXTRA_FEATURES + CORR_FEATURES) if c in df.columns]
miss_by_year = df.groupby('year')[heatmap_features].apply(lambda x: x.isna().mean() * 100)

plt.figure(figsize=(14, 8))
sns.heatmap(miss_by_year.T, cmap='YlOrRd', vmin=0, vmax=100, cbar_kws={'label': '% missing'})
plt.title('Missing values by year (features × year)')
plt.xlabel('Year')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

state_counts = df.groupby('state')['field_id'].nunique()
state_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('husl', len(state_counts)))
axes[0].set_title('Fields per state')
axes[0].set_xlabel('State')
axes[0].set_ylabel('Unique field_id count')
axes[0].tick_params(axis='x', rotation=0)

year_counts = df.groupby('year').size()
year_counts.plot(kind='bar', ax=axes[1], color=sns.color_palette('husl', len(year_counts)))
axes[1].set_title('Rows per year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Row count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()



## 2. Time series — state-level vegetation indices

State mean ± std across all fields per fortnight (main growing-season signal).


In [ ]:

state_ts = (
    df.groupby(['state', 'year', 'fortnight_start'])[VEG_FEATURES]
    .agg(['mean', 'std'])
    .reset_index()
)

fig, axes = plt.subplots(len(VEG_FEATURES), 1, figsize=(14, 3.5 * len(VEG_FEATURES)), sharex=True)
if len(VEG_FEATURES) == 1:
    axes = [axes]

for ax, feat in zip(axes, VEG_FEATURES):
    for state in sorted(df['state'].dropna().unique()):
        sub = state_ts[state_ts['state'] == state]
        ax.plot(sub['fortnight_start'], sub[(feat, 'mean')], label=state, linewidth=2)
        ax.fill_between(
            sub['fortnight_start'],
            sub[(feat, 'mean')] - sub[(feat, 'std')],
            sub[(feat, 'mean')] + sub[(feat, 'std')],
            alpha=0.15,
        )
    ax.set_ylabel(feat)
    ax.legend(loc='upper right')
    ax.set_title(f'State mean {feat} over time (band = ±1 std)')

axes[-1].set_xlabel('Fortnight start')
plt.tight_layout()
plt.show()


In [ ]:

# Facet by year — NDVI per state (Sugarcane-style bar/line comparison, improved for time axis)
g = sns.relplot(
    data=df,
    x='fortnight_start', y='NDVI', hue='state', col='year', col_wrap=3,
    kind='line', estimator='mean', errorbar='sd', height=3.5, aspect=1.6,
    facet_kws={'sharey': False},
)
g.set_axis_labels('Fortnight start', 'Mean NDVI')
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Mean NDVI by state and year')
plt.show()


In [ ]:

# Single-field time series (check one field's fortnight trajectory)
if SAMPLE_FIELD_ID in df['field_id'].values:
    field_df = df[df['field_id'] == SAMPLE_FIELD_ID]
    state_name = field_df['state'].iloc[0]
    fig, ax = plt.subplots(figsize=(14, 5))
    for feat in VEG_FEATURES:
        ax.plot(field_df['fortnight_start'], field_df[feat], marker='o', label=feat, linewidth=2)
    ax.set_title(f'Field {SAMPLE_FIELD_ID} ({state_name}) — vegetation indices')
    ax.set_xlabel('Fortnight start')
    ax.set_ylabel('Index value')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'SAMPLE_FIELD_ID={SAMPLE_FIELD_ID} not in dataset. Pick from:', df['field_id'].unique()[:10])



## 3. Distributions (Sugarcane EDA pattern)


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for ax, feat in zip(axes, VEG_FEATURES):
    sns.histplot(data=df, x=feat, hue='state', kde=True, ax=ax, element='step', stat='density', common_norm=False)
    ax.set_title(f'{feat} distribution by state')
plt.tight_layout()
plt.show()


In [ ]:

# Pairwise relationships — hue by state (replaces label-based pairplot from classification EDA)
pair_cols = VEG_FEATURES + ['state']
sns.pairplot(df[pair_cols].dropna(), hue='state', corner=True, plot_kws={'alpha': 0.4, 's': 12})
plt.suptitle('Vegetation index pairplot by state', y=1.02)
plt.show()



## 4. Correlation heatmap


In [ ]:

corr_cols = [c for c in CORR_FEATURES if c in df.columns]
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation heatmap — key vegetation & moisture features')
plt.tight_layout()
plt.show()



## 5. Data quality — Sentinel-2 coverage


In [ ]:

if 's2_scene_count' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.boxplot(data=df, x='state', y='s2_scene_count', ax=axes[0])
    axes[0].set_title('Sentinel-2 scene count per row by state')

    zero_pct = df.groupby('state').apply(lambda x: (x['s2_scene_count'] == 0).mean() * 100)
    zero_pct.plot(kind='bar', ax=axes[1], color=sns.color_palette('husl', len(zero_pct)))
    axes[1].set_title('% rows with zero S2 scenes')
    axes[1].set_ylabel('%')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()
else:
    print('s2_scene_count column not found — re-run Phase 1 with latest notebook')


In [ ]:

# Optional: MOISTURE & LAI state means over time
moist_cols = [c for c in EXTRA_FEATURES if c in df.columns]
if moist_cols:
    fig, axes = plt.subplots(len(moist_cols), 1, figsize=(14, 4 * len(moist_cols)), sharex=True)
    if len(moist_cols) == 1:
        axes = [axes]
    for ax, feat in zip(axes, moist_cols):
        state_mean = df.groupby(['state', 'fortnight_start'])[feat].mean().reset_index()
        for state in sorted(df['state'].dropna().unique()):
            sub = state_mean[state_mean['state'] == state]
            ax.plot(sub['fortnight_start'], sub[feat], label=state, linewidth=2)
        ax.set_ylabel(feat)
        ax.legend()
        ax.set_title(f'State mean {feat}')
    axes[-1].set_xlabel('Fortnight start')
    plt.tight_layout()
    plt.show()
